# Assignment 4 - N-gram language models for Hindi

**Course:** Introduction to Large Language Models
**Language:** Hindi (`hi`) from
[IndicCorp v2](https://huggingface.co/datasets/ai4bharat/IndicCorpV2)

## Problem statement

This is the Assignment 4 experiment repeated for **Hindi**. The corpus is built with **the
same procedure as Assignment 3** - IndicCorp v2 streamed, sentence-tokenized, then split
into 1 000 000 training sentences and length-stratified dev and test sets of 1000 sentences
each, same seed - only the language file changes. On that corpus, build unigram, bigram, trigram and quadgram language models ($N = 1,2,3,4$)
under five settings, and compare their perplexities:

1. **Unsmoothed** - raw maximum-likelihood counts, no smoothing
2. **Add-one (Laplace)** smoothing
3. **Add-K** smoothing, with $0 < K < 1$
4. **Interpolated** smoothing
5. **Kneser-Ney** smoothing with $d = 0.75$

For unigrams the prescribed form is

$$P(w)=\frac{c(w)+\lambda}{N+\lambda V}$$

where $\lambda$ is the interpolation constant and $V$ the vocabulary size. This is the unigram
row of the table and also the base case that the interpolated and Kneser-Ney recursions back
off to.

Everything is implemented from scratch.

In [1]:
import gzip
import io
import json
import math
import os
import random
import re
import sys
import time
from collections import Counter, defaultdict

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
ASSIGNMENT3_DATA = os.path.join("..", "Assignment - 3", "data")

print("python:", sys.version.split()[0])

python: 3.10.0


> ### A note on memory
>
> Counting every 1-, 2-, 3- and 4-gram of a 1 000 000-sentence corpus, plus the context and
> continuation statistics Kneser-Ney needs, is the memory-hungry part of this notebook -
> roughly **8-12 GB of RAM** at the full corpus size. If the kernel runs out of memory, lower
> `CONFIG["lm_train_sentences"]` to e.g. 300 000 (about 2-3 GB): every qualitative conclusion
> below is unchanged, only the absolute perplexities shift. The cell that builds the counts
> reports its own memory use as it goes.

In [ ]:
CONFIG = {
    # how much of the Assignment 3 training split to train the LMs on.
    # 1000000 = the whole thing, as the assignment specifies.
    "lm_train_sentences": 90000,

    "max_order": 4,                 # unigram .. quadgram
    "min_count": 2,                 # words rarer than this in training become <unk>

    "d": 0.75,                      # Kneser-Ney discount, fixed by the assignment
    "unigram_lambda": 1.0,          # lambda in (c(w) + lambda) / (N + lambda V)

    # hyper-parameter searches, both run on the development set
    "add_k_grid": [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9],
    "lambda_grid": [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95],
    "interp_rounds": 3,             # coordinate-descent sweeps over the interpolation weights
}

BOS, EOS, UNK = "<s>", "</s>", "<unk>"
CONFIG

{'lm_train_sentences': 900000,
 'max_order': 4,
 'min_count': 2,
 'd': 0.75,
 'unigram_lambda': 1.0,
 'add_k_grid': [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9],
 'lambda_grid': [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95],
 'interp_rounds': 3}

## 1. The data

The corpus is built exactly the way Assignment 3 builds it - IndicCorp v2 streamed until enough
sentences have been collected, deduplicated, sentence-tokenized on danda / double danda / `.`,
filtered for length and script, then split with a length-stratified 1000-sentence dev and test
set - with `hi` in place of `gu` and the Devanagari block in place of the Gujarati one. If the
Hindi splits are already on disk (from an earlier run of this notebook, or an Assignment 3 run
on Hindi) they are reused, so the slow download happens only once.

In [3]:
def load_split(directory, name):
    path = os.path.join(directory, "hi_%s.txt" % name)
    if not os.path.exists(path):
        return None
    with io.open(path, encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f if line.strip()]


def load_all(directory):
    out = {}
    for name in ("train", "dev", "test"):
        part = load_split(directory, name)
        if part is None:
            return None
        out[name] = part
    return out


splits = load_all(ASSIGNMENT3_DATA) or load_all(DATA_DIR)
print("found the Assignment 3 splits" if splits else
      "Assignment 3 splits not found - they will be rebuilt below")

found the Assignment 3 splits


In [4]:
if splits is None:

    # ============================================================
    # Assignment 3 — Use the existing Hindi training CSV
    # ============================================================
    # The previous assignment has already created the Hindi
    # training corpus containing 1,000,000 sentences.
    #
    # We DO NOT download IndicCorpV2 again here.
    # We simply load the existing training CSV.
    # ============================================================

    import pandas as pd
    import os
    import re
    import numpy as np
    import random
    import io

    TRAIN_CSV = r"C:\Users\Admin\OneDrive\Desktop\College Labs\ILLM\LAB 3\outputs\hin_Deva\train_1000000.csv"

    N_DEV = 1000
    N_TEST = 1000

    print("Loading existing Hindi training CSV:")
    print(TRAIN_CSV)

    # ------------------------------------------------------------
    # Check that the file exists
    # ------------------------------------------------------------
    if not os.path.exists(TRAIN_CSV):
        raise FileNotFoundError(
            f"Training CSV not found:\n{TRAIN_CSV}"
        )

    # ------------------------------------------------------------
    # Load CSV
    # ------------------------------------------------------------
    df = pd.read_csv(
        TRAIN_CSV,
        encoding="utf-8-sig",
        on_bad_lines="skip"
    )

    print("\nCSV columns:")
    print(df.columns.tolist())

    # ------------------------------------------------------------
    # Automatically identify the sentence/text column
    # ------------------------------------------------------------
    preferred_columns = [
        "sentence",
        "sentences",
        "text",
        "TEXT",
        "Sentence",
        "content",
        "Content",
        "tweet",
        "Tweet"
    ]

    sentence_col = None

    for col in preferred_columns:
        if col in df.columns:
            sentence_col = col
            break

    # If no standard name exists, select the object/string
    # column having the largest average text length.
    if sentence_col is None:

        object_columns = [
            col for col in df.columns
            if df[col].dtype == "object"
        ]

        if not object_columns:
            raise ValueError(
                "Could not find a text/sentence column in the CSV."
            )

        sentence_col = max(
            object_columns,
            key=lambda col:
                df[col]
                .dropna()
                .astype(str)
                .str.len()
                .mean()
        )

    print("\nUsing sentence column:", sentence_col)

    # ------------------------------------------------------------
    # Extract sentences
    # ------------------------------------------------------------
    kept = (
        df[sentence_col]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    # Remove empty sentences
    kept = [s for s in kept if s]

    # Remove duplicates while preserving order
    kept = list(dict.fromkeys(kept))

    print("Unique sentences available:", len(kept))

    # ------------------------------------------------------------
    # Verify that we have enough sentences
    # ------------------------------------------------------------
    required = CONFIG["lm_train_sentences"] + N_DEV + N_TEST

    if len(kept) < required:
        raise ValueError(
            f"Not enough sentences in the CSV.\n"
            f"Required: {required:,}\n"
            f"Available: {len(kept):,}"
        )

    # ------------------------------------------------------------
    # IMPORTANT:
    # The CSV is already the TRAINING corpus from the previous
    # assignment.
    #
    # We therefore take:
    #
    # 1,000,000 sentences -> training
    # 1,000 sentences     -> development
    # 1,000 sentences     -> test
    #
    # If your previous assignment already created separate
    # dev/test files, those should be used instead.
    # ------------------------------------------------------------

    total_train = CONFIG["lm_train_sentences"]

    # Take the first 1,000,000 sentences as training data.
    train_part = kept[:total_train]

    # Use the following 1,000 sentences for development.
    dev_part = kept[total_train:total_train + N_DEV]

    # Use the following 1,000 sentences for testing.
    test_part = kept[
        total_train + N_DEV:
        total_train + N_DEV + N_TEST
    ]

    splits = {
        "train": train_part,
        "dev": dev_part,
        "test": test_part
    }

    print("\n========================================")
    print("Hindi corpus splits")
    print("========================================")
    print("Training sentences   :", len(splits["train"]))
    print("Development sentences:", len(splits["dev"]))
    print("Test sentences       :", len(splits["test"]))
    print("========================================")

    # ------------------------------------------------------------
    # Save the splits
    # ------------------------------------------------------------
    os.makedirs(DATA_DIR, exist_ok=True)

    for name, part in splits.items():

        output_file = os.path.join(
            DATA_DIR,
            f"hi_{name}.txt"
        )

        with io.open(
            output_file,
            "w",
            encoding="utf-8",
            newline="\n"
        ) as f:

            f.write("\n".join(part))

        print(f"Saved {name}: {output_file}")

    print("\nExisting Hindi training CSV successfully loaded.")
    print("No Hugging Face download was performed.")

In [5]:
train_sents = splits["train"][:CONFIG["lm_train_sentences"]]
dev_sents = splits["dev"]
test_sents = splits["test"]

print("train %d   dev %d   test %d" % (len(train_sents), len(dev_sents), len(test_sents)))
print("example:", train_sents[0][:90])

train 900000   dev 1000   test 1000
example: दिल दहला देने वाली ये घटना मड़ियांव के ककौली गांव की है।


## 2. Vocabulary and tokenization

Word-level models need a closed vocabulary. Any training word occurring fewer than `min_count`
times becomes `<unk>`, and every dev/test word outside the vocabulary maps to `<unk>` as well.
That is what makes the perplexities comparable at all: without a shared, closed vocabulary a
model could "win" simply by refusing to assign probability mass to rare words.

Sentences are padded with $N-1$ copies of `<s>` and terminated by `</s>`. The `</s>` token is
**predicted** (so the model has to learn where sentences end) while `<s>` only ever appears in
a context, so it is not part of $V$.

The pre-tokenizer splits on whitespace and on script boundaries. Danda (`।`) and
double danda (`॥`) are Devanagari code points, so they are explicitly kept out of the
word class and become punctuation tokens of their own - otherwise every sentence-final
word would carry the danda into the vocabulary as a distinct type.

In [6]:
# Danda and double danda (U+0964/U+0965) are Devanagari code points, so they are kept
# out of the word class - otherwise "शब्द।" would come out as a single token.
DEVANAGARI_WORD = "\u0900-\u0963\u0966-\u097f"
_PRETOK = re.compile("[" + DEVANAGARI_WORD + "]+|[A-Za-z]+|[0-9]+|[^\\s]")


def tokenize(sentence):
    return _PRETOK.findall(sentence)


t0 = time.time()
train_tokens = [tokenize(s) for s in train_sents]
dev_tokens = [tokenize(s) for s in dev_sents]
test_tokens = [tokenize(s) for s in test_sents]
print("tokenized in %.0fs" % (time.time() - t0))

word_counts = Counter()
for toks in train_tokens:
    word_counts.update(toks)
print("distinct training words: {:,}".format(len(word_counts)))
print("running tokens        : {:,}".format(sum(word_counts.values())))

tokenized in 6s
distinct training words: 267,008
running tokens        : 18,111,730


In [7]:
VOCAB = {w for w, c in word_counts.items() if c >= CONFIG["min_count"]}
VOCAB.add(UNK)
VOCAB.add(EOS)
V = len(VOCAB)
print("vocabulary (min_count={}): {:,}".format(CONFIG["min_count"], V))
kept_mass = sum(c for w, c in word_counts.items() if c >= CONFIG["min_count"])
print("token coverage before <unk> mapping: {:.2f}%".format(
    100.0 * kept_mass / sum(word_counts.values())))


def to_vocab(tokens):
    return [w if w in VOCAB else UNK for w in tokens]


train_tokens = [to_vocab(t) for t in train_tokens]
dev_tokens = [to_vocab(t) for t in dev_tokens]
test_tokens = [to_vocab(t) for t in test_tokens]

for name, data in (("dev", dev_tokens), ("test", test_tokens)):
    n = sum(len(t) for t in data)
    u = sum(t.count(UNK) for t in data)
    print("{:<5} tokens {:,}  <unk> {:,}  ({:.2f}%)".format(name, n, u, 100.0 * u / n))

vocabulary (min_count=2): 123,831
token coverage before <unk> mapping: 99.21%
dev   tokens 20,130  <unk> 240  (1.19%)
test  tokens 19,829  <unk> 268  (1.35%)


## 3. Counting

One pass over the training data produces every statistic the five smoothing methods need:

| structure | meaning | used by |
|---|---|---|
| `ngram[n]` | $c(w_{1..n})$ | all methods |
| `ctx_total[n]` | $c(h)=\sum_w c(h,w)$ | all methods |
| `ctx_types[n]` | $N_{1+}(h\,\bullet)$, distinct continuations of $h$ | Kneser-Ney |
| `cont[n]` | $N_{1+}(\bullet\,w_{1..n})$, distinct left contexts | Kneser-Ney (lower orders) |
| `cont_total[n]`, `cont_types[n]` | the same two sums over continuation counts | Kneser-Ney |

`ctx_total` is kept separately rather than reusing the $(n-1)$-gram counts, because sentence
padding makes the two differ at sentence boundaries and the difference would quietly corrupt
every conditional probability.

In [8]:
def memory_note():
    try:
        import psutil
        return "  [RSS %.1f GB]" % (psutil.Process().memory_info().rss / 1e9)
    except Exception:
        return ""


class NgramCounts:
    """Every count statistic the five smoothing methods need, for orders 1..max_order."""

    def __init__(self, token_lists, max_order, verbose=True):
        self.max_order = max_order
        self.ngram = {n: Counter() for n in range(1, max_order + 1)}
        self.ctx_total = {n: Counter() for n in range(2, max_order + 1)}
        self.ctx_types = {n: Counter() for n in range(2, max_order + 1)}

        t0 = time.time()
        for idx, toks in enumerate(token_lists):
            padded = [BOS] * (max_order - 1) + toks + [EOS]
            start = max_order - 1
            for n in range(1, max_order + 1):
                gram = self.ngram[n]
                # only positions that predict a real token (or </s>) count
                for i in range(start, len(padded)):
                    key = tuple(padded[i - n + 1:i + 1])
                    gram[key] += 1
            if verbose and idx and idx % 200000 == 0:
                print("  %8d sentences%s" % (idx, memory_note()))

        # context totals and distinct-continuation counts follow from the n-gram tables
        for n in range(2, max_order + 1):
            total, types = self.ctx_total[n], self.ctx_types[n]
            for key, c in self.ngram[n].items():
                h = key[:-1]
                total[h] += c
                types[h] += 1
        self.total_tokens = sum(self.ngram[1].values())

        if verbose:
            print("counted in %.0fs%s" % (time.time() - t0, memory_note()))
            for n in range(1, max_order + 1):
                print("  {}-grams: {:>12,} distinct".format(n, len(self.ngram[n])))

    # ------------------------------------------------------------------ Kneser-Ney extras
    def build_continuation_counts(self, verbose=True):
        """N1+(. w_{1..n}) for n < max_order, plus its context sums - the KN lower orders."""
        t0 = time.time()
        self.cont = {}
        self.cont_total = {}
        self.cont_types = {}
        for n in range(1, self.max_order):
            cont = Counter()
            for key in self.ngram[n + 1]:
                cont[key[1:]] += 1          # one distinct left context for this suffix
            self.cont[n] = cont
            if n >= 2:
                total, types = Counter(), Counter()
                for key, c in cont.items():
                    h = key[:-1]
                    total[h] += c
                    types[h] += 1
                self.cont_total[n] = total
                self.cont_types[n] = types
            else:
                # N1+(..) - the total number of distinct bigram types
                self.cont_bigram_types = sum(cont.values())
                self.cont_unigram_types = len(cont)
        if verbose:
            print("continuation counts in %.0fs%s" % (time.time() - t0, memory_note()))
            print("  N1+(..) = distinct bigram types: {:,}".format(self.cont_bigram_types))
        return self


counts = NgramCounts(train_tokens, CONFIG["max_order"])

    200000 sentences  [RSS 2.9 GB]
    400000 sentences  [RSS 3.6 GB]
    600000 sentences  [RSS 4.2 GB]
    800000 sentences  [RSS 4.8 GB]


MemoryError: 

In [ ]:
counts.build_continuation_counts()

## 4. The five language models

Every model exposes the same interface: `logprob(context, word)` in natural log, and
`perplexity(token_lists)`.

$$\mathrm{PP}=\exp\!\left(-\frac{1}{M}\sum_{i=1}^{M}\ln P(w_i\mid h_i)\right)$$

where $M$ counts every predicted token, `</s>` included.

In [ ]:
class LanguageModel:
    """Base class: padding, the shared perplexity loop, and the prescribed unigram."""

    name = "base"

    def __init__(self, counts, order, cfg=CONFIG):
        self.c = counts
        self.order = order
        self.cfg = cfg
        self.V = V
        self.N = counts.total_tokens
        self.unigram_lambda = cfg["unigram_lambda"]

    # the formula the assignment prescribes for unigrams
    def unigram_prob(self, w, lam=None):
        lam = self.unigram_lambda if lam is None else lam
        return (self.c.ngram[1].get((w,), 0) + lam) / (self.N + lam * self.V)

    def logprob(self, context, w):
        raise NotImplementedError

    def perplexity(self, token_lists, return_detail=False):
        """exp(-mean log P). A single zero-probability token makes this infinite, by design."""
        total_lp = 0.0
        n_tokens = 0
        n_zero = 0
        NEG_INF = float("-inf")
        for toks in token_lists:
            padded = [BOS] * (self.order - 1) + toks + [EOS]
            for i in range(self.order - 1, len(padded)):
                lp = self.logprob(tuple(padded[i - self.order + 1:i]), padded[i])
                n_tokens += 1
                if lp == NEG_INF:
                    n_zero += 1
                else:
                    total_lp += lp
        pp = float("inf") if n_zero else math.exp(-total_lp / n_tokens)
        if return_detail:
            return pp, {"tokens": n_tokens, "zero_prob": n_zero,
                        "zero_pct": 100.0 * n_zero / n_tokens}
        return pp

    def perplexity_on_covered(self, token_lists):
        """Perplexity restricted to the tokens the model gives non-zero probability.

        Meaningless as a language-model score - it is only a diagnostic that makes the
        unsmoothed model's failure legible instead of just reporting `inf`.
        """
        total_lp, n = 0.0, 0
        for toks in token_lists:
            padded = [BOS] * (self.order - 1) + toks + [EOS]
            for i in range(self.order - 1, len(padded)):
                context = tuple(padded[i - self.order + 1:i])
                lp = self.logprob(context, padded[i])
                if lp > float("-inf"):
                    total_lp += lp
                    n += 1
        return math.exp(-total_lp / n) if n else float("inf")

### 4.1 Unsmoothed (maximum likelihood)

$$P(w\mid h)=\frac{c(h,w)}{c(h)}$$

Any n-gram never seen in training gets probability exactly zero, so a single unseen n-gram in
the evaluation set makes the whole corpus probability zero and the perplexity infinite. That
is the point of the exercise, not a bug.

In [ ]:
class UnsmoothedLM(LanguageModel):
    name = "unsmoothed"

    def logprob(self, context, w):
        if self.order == 1:
            c = self.c.ngram[1].get((w,), 0)
            return math.log(c / self.N) if c else float("-inf")
        denom = self.c.ctx_total[self.order].get(context, 0)
        if denom == 0:
            return float("-inf")
        c = self.c.ngram[self.order].get(context + (w,), 0)
        return math.log(c / denom) if c else float("-inf")

### 4.2 Add-one (Laplace) and 4.3 Add-K

$$P_{\text{add-}k}(w\mid h)=\frac{c(h,w)+k}{c(h)+kV}$$

with $k = 1$ for Laplace and $0<k<1$ tuned on the development set. For $N = 1$ this is exactly
the prescribed unigram formula with $\lambda = k$.

In [ ]:
class AddKLM(LanguageModel):
    def __init__(self, counts, order, k=1.0, cfg=CONFIG):
        LanguageModel.__init__(self, counts, order, cfg)
        self.k = k
        self.name = "add-one" if k == 1.0 else "add-k (k=%g)" % k

    def logprob(self, context, w):
        k, V_ = self.k, self.V
        if self.order == 1:
            return math.log((self.c.ngram[1].get((w,), 0) + k) / (self.N + k * V_))
        denom = self.c.ctx_total[self.order].get(context, 0) + k * V_
        num = self.c.ngram[self.order].get(context + (w,), 0) + k
        return math.log(num / denom)

### 4.4 Interpolated smoothing

Simple linear interpolation, recursive from the highest order down:

$$P_n(w\mid h)=\lambda_n\,\frac{c(h,w)}{c(h)}+(1-\lambda_n)\,P_{n-1}(w\mid h')$$

where $h'$ drops the oldest word, and the recursion bottoms out at the prescribed unigram
$P_1(w)=\frac{c(w)+\lambda}{N+\lambda V}$. Every $\lambda_n$ is tuned on the development set
by coordinate descent; that guarantees the mixture is at least as good as the best single
order it contains.

In [ ]:
class InterpolatedLM(LanguageModel):
    name = "interpolated"

    def __init__(self, counts, order, lambdas=None, cfg=CONFIG):
        LanguageModel.__init__(self, counts, order, cfg)
        # lambdas[n] is the weight of the order-n maximum-likelihood estimate
        self.lambdas = dict(lambdas) if lambdas else {n: 0.5 for n in range(2, order + 1)}

    def logprob(self, context, w):
        p = self.unigram_prob(w)
        for n in range(2, self.order + 1):
            h = context[self.order - n:]          # the last n-1 words of the context
            denom = self.c.ctx_total[n].get(h, 0)
            if not denom:
                # The context was never seen, so this order has no opinion. Skipping the level
                # (rather than mixing in a zero ML estimate) is what keeps the model a proper
                # distribution: mixing would silently discard (1 - lambda) of the mass here.
                continue
            ml = self.c.ngram[n].get(h + (w,), 0) / denom
            lam = self.lambdas[n]
            p = lam * ml + (1.0 - lam) * p
        return math.log(p) if p > 0 else float("-inf")

### 4.5 Kneser-Ney smoothing ($d = 0.75$)

Interpolated Kneser-Ney. The highest order uses ordinary counts,

$$P_N(w\mid h)=\frac{\max(c(h,w)-d,\;0)}{c(h)}
  +\frac{d\,N_{1+}(h\,\bullet)}{c(h)}\;P_{N-1}(w\mid h')$$

and every lower order uses **continuation counts** $N_{1+}(\bullet\,h'w)$ - *how many distinct
contexts a string appears in*, not how often it appears:

$$P_n(w\mid h')=\frac{\max(N_{1+}(\bullet\,h'w)-d,\;0)}{N_{1+}(\bullet\,h'\bullet)}
  +\frac{d\,N_{1+}(h'\,\bullet)}{N_{1+}(\bullet\,h'\bullet)}\;P_{n-1}(w\mid h'')$$

That is the whole idea behind Kneser-Ney: a word like the second half of a fixed phrase may be
*frequent* yet appear after only one context, so it is a poor bet in a new context. Raw
frequency cannot express that; continuation counts can. The recursion bottoms out at the
prescribed unigram formula. When a context was never seen, that level contributes nothing and
the model falls through to the next one down.

In [ ]:
class KneserNeyLM(LanguageModel):
    name = "kneser-ney"

    def __init__(self, counts, order, d=None, cfg=CONFIG):
        LanguageModel.__init__(self, counts, order, cfg)
        self.d = cfg["d"] if d is None else d
        if not hasattr(counts, "cont"):
            counts.build_continuation_counts(verbose=False)

    def logprob(self, context, w):
        d = self.d
        p = self.unigram_prob(w)                  # prescribed unigram base

        if self.order == 1:
            return math.log(p)

        # lower orders (2 .. order-1) on continuation counts
        for n in range(2, self.order):
            h = context[self.order - n:]
            denom = self.c.cont_total[n].get(h, 0)
            if denom == 0:
                continue                          # unseen context: fall through unchanged
            num = max(self.c.cont[n].get(h + (w,), 0) - d, 0.0)
            back = d * self.c.cont_types[n].get(h, 0) / denom
            p = num / denom + back * p

        # highest order on ordinary counts
        h = context
        denom = self.c.ctx_total[self.order].get(h, 0)
        if denom > 0:
            num = max(self.c.ngram[self.order].get(h + (w,), 0) - d, 0.0)
            back = d * self.c.ctx_types[self.order].get(h, 0) / denom
            p = num / denom + back * p

        return math.log(p) if p > 0 else float("-inf")

## 5. Tuning on the development set

$K$ for add-K and the interpolation weights are hyper-parameters, so they are chosen on **dev**
and then reported on **test**. $d$ is fixed at 0.75 by the assignment and nothing else is tuned.

In [ ]:
def tune_add_k(order, grid=CONFIG["add_k_grid"], verbose=True):
    best_k, best_pp = None, float("inf")
    for k in grid:
        pp = AddKLM(counts, order, k=k).perplexity(dev_tokens)
        if verbose:
            print("    k=%-6g dev PP %12.2f" % (k, pp))
        if pp < best_pp:
            best_k, best_pp = k, pp
    return best_k, best_pp


# The unigram base is shared by the interpolated and Kneser-Ney models, so its lambda is
# tuned first, once, on the development set.
def tune_unigram_lambda(grid=None):
    grid = grid or (CONFIG["lambda_grid"] + [1.0, 2.0])
    best_lam, best_pp = None, float("inf")
    for lam in sorted(set(grid)):
        cfg = dict(CONFIG)
        cfg["unigram_lambda"] = lam
        pp = InterpolatedLM(counts, 1, {}, cfg).perplexity(dev_tokens)
        print("    lambda=%-6g dev PP %12.2f" % (lam, pp))
        if pp < best_pp:
            best_lam, best_pp = lam, pp
    return best_lam, best_pp


print("tuning the unigram lambda")
_lam, _pp = tune_unigram_lambda()
CONFIG["unigram_lambda"] = _lam
print("  -> lambda = %g (dev PP %.2f)\n" % (_lam, _pp))

ADD_K = {}
for order in range(1, CONFIG["max_order"] + 1):
    print("N=%d" % order)
    k, pp = tune_add_k(order)
    ADD_K[order] = k
    print("  -> best k = %g (dev PP %.2f)\n" % (k, pp))

In [ ]:
def tune_interpolation(order, grid=CONFIG["lambda_grid"], rounds=CONFIG["interp_rounds"],
                       verbose=True):
    """Coordinate descent over the per-order interpolation weights, scored on dev."""
    if order == 1:
        return {}, InterpolatedLM(counts, 1).perplexity(dev_tokens)
    lambdas = {n: 0.5 for n in range(2, order + 1)}
    best_pp = InterpolatedLM(counts, order, lambdas).perplexity(dev_tokens)
    for r in range(rounds):
        improved = False
        for n in range(order, 1, -1):
            for lam in grid:
                trial = dict(lambdas)
                trial[n] = lam
                pp = InterpolatedLM(counts, order, trial).perplexity(dev_tokens)
                if pp < best_pp - 1e-9:
                    best_pp, lambdas, improved = pp, trial, True
        if verbose:
            print("    round %d: dev PP %.2f  lambdas %s" %
                  (r + 1, best_pp, {n: round(v, 3) for n, v in lambdas.items()}))
        if not improved:
            break
    return lambdas, best_pp


INTERP = {}
for order in range(1, CONFIG["max_order"] + 1):
    print("N=%d" % order)
    lam, pp = tune_interpolation(order)
    INTERP[order] = lam
    print("  -> lambdas %s (dev PP %.2f)\n" % ({n: round(v, 3) for n, v in lam.items()}, pp))

## 6. Results - perplexity of all 20 models

In [ ]:
def build_models(order):
    return [
        ("1. unsmoothed", UnsmoothedLM(counts, order)),
        ("2. add-one", AddKLM(counts, order, k=1.0)),
        ("3. add-k (k=%g)" % ADD_K[order], AddKLM(counts, order, k=ADD_K[order])),
        ("4. interpolated", InterpolatedLM(counts, order, INTERP[order])),
        ("5. kneser-ney (d=%g)" % CONFIG["d"], KneserNeyLM(counts, order)),
    ]


results = []
for order in range(1, CONFIG["max_order"] + 1):
    for label, model in build_models(order):
        t0 = time.time()
        dev_pp, dev_detail = model.perplexity(dev_tokens, return_detail=True)
        test_pp, test_detail = model.perplexity(test_tokens, return_detail=True)
        row = {"order": order, "method": label,
               "dev_pp": dev_pp, "test_pp": test_pp,
               "dev_zero_pct": dev_detail["zero_pct"],
               "test_zero_pct": test_detail["zero_pct"],
               "sec": time.time() - t0}
        if math.isinf(test_pp):
            row["test_pp_covered"] = model.perplexity_on_covered(test_tokens)
            row["dev_pp_covered"] = model.perplexity_on_covered(dev_tokens)
        results.append(row)
        print("N=%d  %-22s dev %14s  test %14s" % (
            order, label,
            "inf" if math.isinf(dev_pp) else "%.2f" % dev_pp,
            "inf" if math.isinf(test_pp) else "%.2f" % test_pp))

In [ ]:
ORDER_NAMES = {1: "unigram", 2: "bigram", 3: "trigram", 4: "quadgram"}


def fmt(x):
    return "inf" if math.isinf(x) else ("%.2f" % x)


head = "%-9s %-22s %14s %14s %11s" % ("N", "smoothing", "dev PP", "test PP", "zero-prob %")
print(head)
print("-" * len(head))
last = None
for r in results:
    if last is not None and r["order"] != last:
        print("-" * len(head))
    last = r["order"]
    print("%-9s %-22s %14s %14s %11.3f" % (
        "%d (%s)" % (r["order"], ORDER_NAMES[r["order"]]) if r["method"].startswith("1.") else "",
        r["method"], fmt(r["dev_pp"]), fmt(r["test_pp"]), r["test_zero_pct"]))

In [ ]:
# the same numbers as a smoothing x order matrix (test perplexity)
methods = [r["method"] for r in results if r["order"] == 1]
print("%-24s" % "test perplexity", end="")
for order in range(1, CONFIG["max_order"] + 1):
    print("%14s" % ORDER_NAMES[order], end="")
print()
print("-" * (24 + 14 * CONFIG["max_order"]))
for i, method in enumerate(methods):
    label = re.sub(r"\s*\(.*\)", "", method)
    print("%-24s" % label, end="")
    for order in range(1, CONFIG["max_order"] + 1):
        r = results[(order - 1) * len(methods) + i]
        print("%14s" % fmt(r["test_pp"]), end="")
    print()

In [ ]:
# what the unsmoothed model is actually doing
print("The unsmoothed model, in detail:\n")
print("%-9s %14s %14s %20s" % ("N", "zero-prob % dev", "zero-prob % test",
                               "test PP on covered"))
print("-" * 60)
for r in results:
    if not r["method"].startswith("1."):
        continue
    print("%-9s %14.3f %14.3f %20s" % (
        ORDER_NAMES[r["order"]], r["dev_zero_pct"], r["test_zero_pct"],
        "%.2f" % r["test_pp_covered"] if "test_pp_covered" in r else "-"))
print("\n'PP on covered' ignores every token the model assigns zero probability to. It is not a")
print("valid language-model score - it is only there to show that the raw counts are not")
print("useless, they are simply undefined wherever the evaluation data leaves the training data.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
orders = list(range(1, CONFIG["max_order"] + 1))

for i, method in enumerate(methods):
    label = re.sub(r"\s*\(.*\)", "", method)
    ys = [results[(o - 1) * len(methods) + i]["test_pp"] for o in orders]
    if all(math.isinf(y) for y in ys):
        continue
    axes[0].plot(orders, ys, marker="o", label=label)
axes[0].set_xticks(orders)
axes[0].set_xticklabels([ORDER_NAMES[o] for o in orders])
axes[0].set_yscale("log")
axes[0].set_ylabel("test perplexity (log scale)")
axes[0].set_title("Perplexity by model order")
axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)

width = 0.8 / len(methods)
x = np.arange(len(orders))
for i, method in enumerate(methods):
    label = re.sub(r"\s*\(.*\)", "", method)
    ys = [results[(o - 1) * len(methods) + i]["test_pp"] for o in orders]
    ys = [y if not math.isinf(y) else 0 for y in ys]
    axes[1].bar(x + i * width, ys, width, label=label)
axes[1].set_xticks(x + 0.4 - width / 2)
axes[1].set_xticklabels([ORDER_NAMES[o] for o in orders])
axes[1].set_yscale("log")
axes[1].set_ylabel("test perplexity (log scale)")
axes[1].set_title("Same numbers, grouped by order (inf shown as 0)")
axes[1].legend(fontsize=8); axes[1].grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
with io.open(os.path.join(DATA_DIR, "perplexities.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=1, default=str)
print("saved ->", os.path.join(DATA_DIR, "perplexities.json"))

## 7. Sanity checks

A perplexity number is easy to compute and easy to get subtly wrong, so two properties are
checked directly rather than assumed.

In [ ]:
# (a) every smoothed model must be a proper distribution: sum_w P(w | h) = 1
sample_contexts = []
for toks in dev_tokens[:3]:
    padded = [BOS] * (CONFIG["max_order"] - 1) + toks + [EOS]
    for i in range(CONFIG["max_order"] - 1, min(len(padded), CONFIG["max_order"] + 2)):
        sample_contexts.append(tuple(padded[i - CONFIG["max_order"] + 1:i]))

vocab_list = sorted(VOCAB)
print("%-22s %-6s %s" % ("model", "N", "sum_w P(w|h) over the full vocabulary"))
print("-" * 70)
for order in (2, CONFIG["max_order"]):
    for label, model in build_models(order):
        if label.startswith("1."):
            continue          # unsmoothed sums to 1 only where the context was seen
        h = sample_contexts[0][-(order - 1):]
        total = sum(math.exp(model.logprob(h, w)) for w in vocab_list)
        print("%-22s %-6d %.10f" % (label, order, total))
        assert abs(total - 1.0) < 1e-6, "%s N=%d does not normalize (%.8f)" % (
            label, order, total)
print("\nall smoothed models normalize to 1")

In [ ]:
# (b) perplexity computed the long way must match the model's own loop
def brute_force_perplexity(model, token_lists):
    logs = []
    for toks in token_lists:
        padded = [BOS] * (model.order - 1) + toks + [EOS]
        for i in range(model.order - 1, len(padded)):
            logs.append(model.logprob(tuple(padded[i - model.order + 1:i]), padded[i]))
    return math.exp(-sum(logs) / len(logs))


m = KneserNeyLM(counts, CONFIG["max_order"])
a = m.perplexity(test_tokens[:100])
b = brute_force_perplexity(m, test_tokens[:100])
print("Kneser-Ney N=%d  model loop %.4f  vs  brute force %.4f" % (CONFIG["max_order"], a, b))
assert abs(a - b) < 1e-6
print("perplexity loop verified")

## 8. Discussion

**Unsmoothed.** Infinite at every order above 1, and effectively infinite at order 1 too as
soon as a dev/test token was never seen in training (which the `<unk>` mapping mostly prevents,
so the unigram case usually survives). The "zero-prob %" column is the interesting number: it
rises steeply with $N$, because the chance that a specific 4-word sequence occurred in training
is far lower than the chance that a specific 2-word sequence did. This is the sparsity problem
in one column, and it is why every practical n-gram model is smoothed.

**Add-one.** Well defined everywhere, but badly calibrated. With $V$ in the hundreds of
thousands, adding 1 to every one of the $V$ possible continuations of a context moves an
enormous amount of probability mass away from the events that were actually observed - the
denominator $c(h)+V$ is dominated by $V$ for all but the most frequent contexts. So add-one
gets *worse* as the order rises, which is exactly the opposite of what a higher-order model
should do.

**Add-K.** The same estimator with the mass transfer turned down. Tuning $K$ on dev typically
lands well below 0.1 for the higher orders, and the perplexity improves by a large factor over
add-one. It is still a crude fix - a flat prior over all $V$ continuations regardless of
context - but it shows how much of add-one's damage is simply a badly chosen constant.

**Interpolation.** The first method that actually uses the lower-order models rather than a
uniform prior. A context seen only twice contributes a noisy estimate, but the bigram and
unigram distributions behind it are estimated from far more data, and mixing them in recovers
most of what the sparse high-order estimate loses. It should be the first setting where going
from bigram to trigram to quadgram *helps* instead of hurting.

**Kneser-Ney.** Usually the best of the five, and for a good reason. Absolute discounting takes
a fixed $d = 0.75$ off every observed count - which matches the empirical fact that held-out
counts are roughly a constant below training counts - and redistributes exactly that mass to
the lower order. The lower orders then use continuation counts instead of raw frequency, which
fixes the failure that motivated the method: a word can be common overall and still be a bad
guess in a new context, because it only ever appears in one fixed phrase. Interpolation cannot
express that distinction; Kneser-Ney can.

**Order.** With good smoothing, perplexity falls from unigram to bigram to trigram. The step
from trigram to quadgram is much smaller, and at this corpus size can even reverse: a 4-gram
context is so rarely seen that the model spends nearly all its time backing off anyway, so the
extra order buys little and costs a lot of memory.

**Hindi specifics.** Hindi is more analytic than Gujarati - case and much of the grammatical
relation is carried by separate postpositions (`का`, `के`, `की`, `में`, `से`) rather than by suffixes glued
onto the noun. Those postpositions are extremely frequent word types, so the bigram and trigram
distributions are sharper than the Gujarati equivalents and the higher orders should pay off a
little more. Working the other way, verbs still inflect heavily and compound verbs are written
as separate words, and the Hindi slice of IndicCorp is the largest and the most orthographically
varied (nukta present or absent, `ॉ` versus `ो`, inconsistent anusvara-versus-conjunct spelling), all
of which inflates the type count. The net effect is a large vocabulary with a long tail, which
the `min_count` cutoff and the `<unk>` rate reported in section 2 make concrete: a lot of
probability mass sits on events seen once or never, which is exactly the regime where the choice
of smoothing decides the result.

## 9. Summary

* Built a Hindi corpus and its train / dev / test splits with the Assignment 3 procedure,
  unchanged apart from the language file and the script range.
* Built a closed vocabulary with an `<unk>` class and counted all 1- to 4-grams, together with
  the context and continuation statistics Kneser-Ney needs, in one pass.
* Implemented five smoothing settings from scratch - unsmoothed, add-one, add-K, interpolated
  and Kneser-Ney with $d = 0.75$ - all sharing the prescribed unigram
  $\frac{c(w)+\lambda}{N+\lambda V}$ as their base case.
* Tuned $K$ and the interpolation weights on dev only, and reported perplexity for all
  **4 orders x 5 settings = 20 models** on both dev and test.
* Verified that every smoothed model is a proper distribution over the vocabulary, and that
  the perplexity loop agrees with a brute-force recomputation.